# Inference of transcription factor activity

## [decoupleR](https://github.com/saezlab/decoupleR)

> decoupleR is a Bioconductor package containing different statistical methods to extract biological signatures from prior knowledge within a unified framework. Additionally, it incorporates methods that take into account the sign and weight of network interactions.

In [ ]:
# # Note: This is an advanced module requiring the decoupleR and dorothea packages.
# suppressPackageStartupMessages(library(OmnipathR))
# suppressPackageStartupMessages(library(decoupleR))

In [ ]:
# # 1. Load murine TF-target network from DoRothEA (Confidence levels A, B, C)
# net <- get_dorothea(organism = 'mouse', levels = c('A', 'B', 'C'))

In [ ]:
# 2. Run multivariate linear model (mlm) or weighted mean (wmean) to infer activity
# tf_activities <- decoupleR::run_wmean(
#     mat = counts, network = net, .source = 'source', .target = 'target', 
#     .mor = 'mor')
# Activity inference with Univariate Linear Model (ULM)
# tf_activities <- decoupleR::run_ulm(
#     mat = counts, net = net, .source = 'source', .target = 'target', 
#     .mor = 'mor', minsize = 5)

In [ ]:
# 3. Visualize dynamic activities of known pluripotency markers across stages
# (Code here would extract scores for Pou5f1, Nanog, Sox2 and plot them as a heatmap)
# cat("TF Activity module successfully configured for downstream network analysis.\n")

In [10]:
library(tidyverse)
suppressPackageStartupMessages(library(GenomicRanges))

In [3]:
gtf <- file.path("~/reference/Mus_musculus/mm10", "gencode.vM23.annotation.gtf") |>
    rtracklayer::import()

In [9]:
Sys.time()
exons_length <- gtf |> 
    subset(type == "exon") |>
    GenomicRanges::split(~ gene_id) |>
    lapply(reduce) |>
    sapply(FUN = function(gr) sum(width(gr))) |>
    enframe(name = "gene_id", value = "basepairs")
Sys.time()

[1] "2026-06-22 19:10:54 CST"

[1] "2026-06-22 19:15:42 CST"

In [ ]:
anno_gene <- gtf |>
    subset(type == "gene") |>
    subset(gene_type %in% c("protein_coding", "lncRNA")) |>
    as.data.frame() |>
    distinct(
        chr = seqnames, start, end, strand, 
        gene_id, symbol = gene_name, type = gene_type) |>
    left_join(exons_length, by = "gene_id") |>
    mutate(gene_id = sub("\\..*$", "", gene_id)) |>
    column_to_rownames("gene_id") |> 
    GRanges()

anno_gene

GRanges object with 31812 ranges and 3 metadata columns:
                     seqnames          ranges strand |      symbol
                        <Rle>       <IRanges>  <Rle> | <character>
  ENSMUSG00000051951     chr1 3205901-3671498      - |        Xkr4
  ENSMUSG00000089699     chr1 3466587-3513553      + |      Gm1992
  ENSMUSG00000102331     chr1 3647309-3658904      - |     Gm19938
  ENSMUSG00000102343     chr1 3905739-3986215      - |     Gm37381
  ENSMUSG00000025900     chr1 3999557-4409241      - |         Rp1
                 ...      ...             ...    ... .         ...
  ENSMUSG00000065947     chrM      9877-10173      + |     mt-Nd4l
  ENSMUSG00000064363     chrM     10167-11544      + |      mt-Nd4
  ENSMUSG00000064367     chrM     11742-13565      + |      mt-Nd5
  ENSMUSG00000064368     chrM     13552-14070      - |      mt-Nd6
  ENSMUSG00000064370     chrM     14145-15288      + |     mt-Cytb
                               type basepairs
                        <c

In [ ]:
saveRDS(anno_gene, file.path("~/reference/Mus_musculus/mm10", "gencode.vM23.annotation.rds"))

In [1]:
library(tidyverse)
suppressPackageStartupMessages(library(GenomicRanges))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [2]:
gtf <- file.path("~/reference/Homo_sapiens/hg38", "gencode.v48.annotation.gtf") |>
    rtracklayer::import()

Sys.time()
exons_length <- gtf |> 
    subset(type == "exon") |>
    split(~ gene_id) |>
    lapply(reduce) |>
    sapply(FUN = function(gr) sum(width(gr))) |>
    enframe(name = "gene_id", value = "basepairs")
Sys.time()

anno_gene <- gtf |>
    subset(type == "gene") |>
    subset(gene_type %in% c("protein_coding", "lncRNA")) |>
    as.data.frame() |>
    distinct(
        chr = seqnames, start, end, strand, 
        gene_id, symbol = gene_name, type = gene_type) |>
    left_join(exons_length, by = "gene_id") |>
    mutate(gene_id = sub("\\..*$", "", gene_id)) |>
    column_to_rownames("gene_id") |> 
    GRanges()

anno_gene

saveRDS(anno_gene, file.path("~/reference/Homo_sapiens/hg38", "gencode.v48.annotation.rds"))

[1] "2026-06-22 19:16:57 CST"

[1] "2026-06-22 19:23:52 CST"

GRanges object with 54978 ranges and 3 metadata columns:
                  seqnames      ranges strand |          symbol           type
                     <Rle>   <IRanges>  <Rle> |     <character>    <character>
  ENSG00000290825     chr1 11121-24894      + |        DDX11L16         lncRNA
  ENSG00000310526     chr1 14356-30744      - |          WASH7P         lncRNA
  ENSG00000243485     chr1 28589-31109      + |     MIR1302-2HG         lncRNA
  ENSG00000237613     chr1 34553-37595      - |         FAM138A         lncRNA
  ENSG00000308361     chr1 36526-40778      + | ENSG00000308361         lncRNA
              ...      ...         ...    ... .             ...            ...
  ENSG00000212907     chrM 10470-10766      + |         MT-ND4L protein_coding
  ENSG00000198886     chrM 10760-12137      + |          MT-ND4 protein_coding
  ENSG00000198786     chrM 12337-14148      + |          MT-ND5 protein_coding
  ENSG00000198695     chrM 14149-14673      - |          MT-ND6 protein_co